In [27]:
import os, sys, warnings, pandas as pd, psycopg
from dotenv import load_dotenv
sys.path.insert(0, "..")
from collector.embed import clean_for_embedding
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
load_dotenv("../.env")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, connect_timeout=10)

LABELS = "../data/labels"
CODES = {"crash", "wont_launch", "performance", "bugs", "lost_saves", "servers", "accounts_drm", "controls_ui",
         "price_value", "monetization", "dlc", "cheaters_toxicity", "multiplayer", "updates_support", "gen_ai",
         "politics", "boring_grind", "difficulty_combat", "story", "tutorial", "worse_than_predecessor",
         "none", "not_english", "graphics_audio"}
PATH = f"{LABELS}/review_dev_to_label.csv"          # switch to review_test_to_label.csv for the test set

In [33]:
lab = pd.read_csv(PATH, encoding="utf-8-sig", dtype={"topics": str, "note": str})
START = 20                                            # 0, 10, 20, ...
for i, r in lab.iloc[START:START + 10].iterrows():
    print(f"[{i}] {r['name']}\n{r.review}\n{'-' * 100}")

[20] Pathfinder: Wrath of the Righteous - Enhanced Edition
In-your-face politically motivated character relationships being presented within the first few hours of gameplay isn't something I can recommend in any writing in any piece of media. It was annoying and unnecessary tokenization in The Elder Scrolls: Online, and it's annoying as well as unnecessary tokenization here.

Writing, ideally, should be subtle. The incident that inspired this review was about as subtle as high-impact dentistry via mechanical application of masonry.
----------------------------------------------------------------------------------------------------
[21] Borderlands 4
Permanent stutter, all graphics settings have the same performance with the same issue, can't play a game like that.
----------------------------------------------------------------------------------------------------
[22] Supermarket Simulator
really sucks how the devs ruined such a good game, i highly advise against buying this game since

In [35]:
answers = {
    0: ('controls_ui', "control issues, failed tutorial because of controls"),
    1: ('monetization', 'about what the subscription covers'),
    2: ('controls_ui', 'game runs but technology not compatible'),
    3: ('controls_ui', 'control complaint'),
    4: ('monetization;dlc;price_value', 'complaints about developer choices surrounding in-game purchases, adding paid cosmetics is not an update complaints'),
    5: ('difficulty_combat;boring_grind', 'general complaints'),
    6: ('boring_grind;controls_ui', 'missing conveniences'),
    7: ('difficulty_combat;performance', 'unfair, poor graphic optimization'),
    8: ('lost_saves', 'lost save file'),
    9: ('difficulty_combat', 'npcs stronger than player'),
    10: ('bugs', 'game is a scam because of bugs hidden in late game'),
    11: ('boring_grind;story;graphics_audio', 'repetitive gameplay, poor writing, poor music'),
    12: ('controls_ui', 'terrible controls, UI scaling issues for specifically steam deck'),
    13: ('controls_ui;price_value;tutorial', 'clunky controls, not worth price, building is hard and confusing'),
    14: ('boring_grind;tutorial;difficulty_combat;controls_ui', 'poor tutorial, boring, work like gameplay, boring combat'),
    15: ('boring_grind;difficulty_combat;multiplayer', 'boring gameplayloop, expensive kits can be lost to free kits, multiplayer gameplay ruins experience'),
    16: ('gen_ai', 'dislikes use of AI'),
    17: ('politics;story', 'complaint about generic political view, poor story, twists, writing'),
    18: ('servers', 'disconnections'),
    19: ('wont_launch', 'crashes at startup'),
    20: ('politics;story', 'dislike of politics present, writing not subtle enough'),
    21: ('performance', 'all graphics settings have performance issue'),
    22: ('updates_support;dlc', 'devs ruined game, content hidden behind dlc'),
    23: ('updates_support', 'lack of updates for extended period'),
    24: ('multiplayer;story;cheaters_toxicity;boring_grind;graphics_audio;servers;worse_than_predecessor', 'poor multiplayer due to toxicity, backwards gameplay since previous version, boring story, ugly graphics'),
    25: ('monetization;controls_ui;worse_than_predecessor', 'repetitive pop up ads, cannot remove waypoint, worst game in series'),
    26: ('servers', 'densyncing issues'),
    27: ('bugs', 'bug in last mission'),
    28: ('crash', 'frequent crashes'),
    29: ('price_value;worse_than_predecessor;boring_grind;updates_support', 'not worth price, previous release has better gameplay, more value, little to do post story, post launch support boring'),
    30: ('accounts_drm;monetization', 'sign in inconvenience, poor odds on rare character'),
    31: ('dlc;bugs;price_value', 'overpriced DLC, hole/map bugs'),
    32: ('multiplayer;boring_grind;servers;monetization;crash;performance', 'high change of disconnecting and crashing, lack of content, unable to easily access multiplayer gamemodes, vast monetization, time compiling shaders'),
    33: ('gen_ai', 'dislike for use of AI'),
    34: ('crash', 'common crashes'),
    35: ('bugs;updates_support', 'game breaking bugs, new changes without proper testing'),
    36: ('lost_saves', 'loss of save progress'),
    37: ('updates_support', 'complaints about announcement decision'),
    38: ('boring_grind', 'tedious work'),
    39: ('bugs;crash;controls_ui', 'game crashes, common bugs, performs poorely on Xbox, poor item descriptions'),
    40: ('graphics_audio;story;boring_grind', 'poor voice acting, poor dialog, boring game'),
    41: ('cheaters_toxicity;updates_support', 'cheating is common, believed to be abandoned'),
}

done = sorted(answers)
bad = {i: c for i in done for c in answers[i][0].split(";") if c not in CODES}
assert not bad, f"unknown codes: {bad}"
assert all(answers[i][0] for i in done), "use 'none' for no topic, never an empty string"
lab.loc[done, "topics"] = [answers[i][0] for i in done]
lab.loc[done, "note"] = [answers[i][1] for i in done]
lab.to_csv(PATH, index=False, encoding="utf-8-sig")
print(f"{lab.topics.notna().sum()} of {len(lab)} labeled")

42 of 42 labeled


In [36]:
TEST_LABEL, TEST_KEY = f"{LABELS}/review_test_to_label.csv", f"{LABELS}/review_test_key.csv"
SEED, CUTOFF = "20261102", "2026-10-25"
QUOTA = {(False, "20-199"): 34, (False, "200-999"): 33, (False, "1000+"): 33,
         (True, "20-199"): 20, (True, "200-999"): 20, (True, "1000+"): 20}
dev_ids = pd.read_csv(f"{LABELS}/review_dev_key.csv").recommendationid.tolist()

pool = pd.read_sql("""
WITH s AS (
    SELECT r.recommendationid, r.appid, r.voted_up,
           CASE WHEN length(btrim(r.review, E' \\t\\r\\n')) < 200  THEN '20-199'
                WHEN length(btrim(r.review, E' \\t\\r\\n')) < 1000 THEN '200-999'
                ELSE '1000+' END AS length_group
    FROM reviews r
    WHERE NOT r.is_short AND r.created_at < %(cutoff)s::date AND NOT (r.recommendationid = ANY(%(dev)s))
), one_per_game AS (
    SELECT *, row_number() OVER (PARTITION BY voted_up, length_group, appid
                                 ORDER BY md5(recommendationid::text || %(seed)s)) AS nth
    FROM s
)
SELECT recommendationid, appid, voted_up, length_group,
       row_number() OVER (PARTITION BY voted_up, length_group ORDER BY md5(recommendationid::text || %(seed)s)) AS k
FROM one_per_game WHERE nth = 1
""", conn, params={"cutoff": CUTOFF, "dev": dev_ids, "seed": SEED})

test = pd.concat([pool[(pool.voted_up == up) & (pool.length_group == g) & (pool.k <= n)]
                  for (up, g), n in QUOTA.items()])
assert len(test) == sum(QUOTA.values()), test.groupby(["voted_up", "length_group"]).size()

texts = pd.read_sql("""SELECT r.recommendationid, g.name, r.review FROM reviews r JOIN games g USING (appid)
                       WHERE r.recommendationid = ANY(%(ids)s)""", conn, params={"ids": test.recommendationid.tolist()})
test = test.merge(texts, on="recommendationid", validate="one_to_one")

if os.path.exists(TEST_LABEL):
    print("test files already exist; not overwriting")
else:
    test = test.sample(frac=1, random_state=int(SEED))                          # shuffle
    test.sort_values("recommendationid")[["recommendationid", "voted_up", "length_group"]].to_csv(TEST_KEY, index=False)
    (test[["recommendationid", "name", "review"]]
        .assign(review=lambda d: d.review.map(clean_for_embedding), topics=None, note=None)
        .to_csv(TEST_LABEL, index=False, encoding="utf-8-sig"))
    print(f"wrote {len(test)} test reviews")

wrote 160 test reviews


In [56]:
PATH = TEST_LABEL
lab = pd.read_csv(PATH, encoding="utf-8-sig", dtype={"topics": str, "note": str})
print(sorted(CODES))                                 # keep the codes in view
START = 159                                            # 0, 10, 20, ... 150
for i, r in lab.iloc[START:START + 10].iterrows():
    print(f"[{i}] {r['name']}\n{r.review}\n{'-' * 100}")

['accounts_drm', 'boring_grind', 'bugs', 'cheaters_toxicity', 'controls_ui', 'crash', 'difficulty_combat', 'dlc', 'gen_ai', 'graphics_audio', 'lost_saves', 'monetization', 'multiplayer', 'none', 'not_english', 'performance', 'politics', 'price_value', 'servers', 'story', 'tutorial', 'updates_support', 'wont_launch', 'worse_than_predecessor']
[159] CODE VEIN II
This is "Code Vein" in name only. It is more of a reboot rather than a sequel. From what I gathered during the time I spent playing thus far, it has nothing to do with the original game and the new story they are telling is horribly confusing and convoluted, as well as painfully slow-paced as time padding, and thus completely unengaging (although that is subjective). Character creation is more limited than it was in the previous game, companions are about as one-dimensional in the first game, characters run like they made a doo-doo in their diapers, and combat feels like a downgrade from the previous installment, as it feels less

In [58]:
test_answers = {
    0: ("none", "general complaint of playing too long for nothing"),
    1: ("none", "general complaint"),
    2: ("difficulty_combat;bugs;price_value", "poor movement, meh combat, soft locks, get on huge discount"),
    3: ("none", "complaint about specific character"),
    4: ("difficulty_combat", "balancing issues with certain combat mechanic"),
    5: ("none", "general complaint"),
    6: ("none", "positive review"),
    7: ("performance", "call to improve performance by reducing graphics"),
    8: ("multiplayer", "not great for coop"),
    9: ("worse_than_predecessor;difficulty_combat;story;price_value", "downgrade from first game, poor combat, boring story, not worth the prices"),
    10: ("monetization", "good content behind paywalls, not boring_grind, complaint of stone axes to alien transmitters is complaint about the logic in game"),
    11: ("none", "request for specific feature"),
    12: ("none", "positive review"),
    13: ("boring_grind;difficulty_combat;story", "long missions, dumb ai, forced concept is poor writing"),
    14: ("bugs;crash;difficulty_combat", "lots of bugs, crashing common, new gear is not helpful"),
    15: ("none", "positive review, request for specific cosmetic"),
    16: ("difficulty_combat", "positive review, but says challenges can be a bit much"),
    17: ("difficulty_combat", "enemies too strong, certain key mechanic punishes a play style"),
    18: ("none", "positive review"),
    19: ("monetization;cheaters_toxicity", "powerful robots locked behind purchases, cheaters are common, makes combat difficult, not specific complaint about combat"),
    20: ("performance;difficulty_combat;boring_grind", "optimization issues lead to performance drops, combat parts unbalanced, gamemode is too long"),
    21: ("bugs", "glitched login screen"),
    22: ("none", "positive review"),
    23: ("none", "general negative review"),
    24: ("boring_grind", "grindy game"),
    25: ("servers", "low amount of people on servers, online fighting is unfair with server performance"),
    26: ("none", "general character complaint"),
    27: ("bugs", "specific frustrating bug"),
    28: ("worse_than_predecessor;boring_grind", "boring, repetitive missions, lost what made previous games fun"),
    29: ("bugs;price_value", "bugs lock fps, not worth its price"),
    30: ("multiplayer", "dead game"),
    31: ("bugs;boring_grind", "few bugs, can feel tedious"),
    32: ("lost_saves;price_value", "lost progress, want refund, waste of money because of possibility of lost save"),
    33: ("boring_grind", "manual work becomes tedious"),
    34: ("cheaters_toxicity", "poor community within servers, cheaters in servers"),
    35: ("bugs;price_value;worse_than_predecessor", "bugs, less content than previous game, don't buy at full price"),
    36: ("none", "positive review"),
    37: ("bugs;difficulty_combat;performance", "visual stutters, bugs, teleporters could be buffed to make exploration easier"),
    38: ("difficulty_combat;bugs;monetization", "roles are unbalanced, strong meta, glitches, profit focus over community"),
    39: ("monetization", "subscription needed for shop and game features"),
    40: ("accounts_drm", "game unsafe for players, scam game"),
    41: ("none", "positive review"),
    42: ("boring_grind;difficulty_combat", "boring to play, frustrating combat"),
    43: ("boring_grind", "mechanics produce friction and makes game a grind"),
    44: ("none", "positive review"),
    45: ("story;worse_than_predecessor", "story worse than "),
    46: ("boring_grind;difficulty_combat", "boring, grindy game, shallow combat"),
    47: ("difficulty_combat", "frustrating combat"),
    48: ("none", "complaint about specifically female avatar appearance"),
    49: ("none", "positive review"),
    50: ("none", "positive review"),
    51: ("none", "complaint was about the game being too short and not satisfying"),
    52: ("performance;boring_grind;worse_than_predecessor", "performance worse on older systems, shallow game makes it less enjoyable, worse than predecessor"),
    53: ("none", "positive review"),
    54: ("none", "positive review"),
    55: ("performance", "game runs poorely"),
    56: ("difficulty_combat", "poor balancing for update"),
    57: ("none", "positive review"),
    58: ("difficulty_combat", "movement complaint"),
    59: ("boring_grind;graphics_audio;tutorial", "repetitive gameplay, graphics could be better, intro took too long"),
    60: ("servers", "complaint on server limitations"),
    61: ("gen_ai", "complaint on use of generative ai"),
    62: ("difficulty_combat", "enemy AI too strong"),
    63: ("performance;difficulty_combat", "enemy AI too strong, performance issues"),
    64: ("none", "positive review"),
    65: ("none", "vague complaint"),
    66: ("controls_ui;difficulty_combat;story;graphics_audio", "complaint on UI, complaint on certain deaths, poor story, poor graphics, steep difficulty increase"),
    67: ("none", "neither really positive or negative, feature of the game"),
    68: ("boring_grind", "boring gameplay"),
    69: ("worse_than_predecessor;controls_ui;graphics_audio", "dissapointment compared to previous games, poor camera, clanky controls, bad graphics"),
    70: ("controls_ui", "clunky movement and controls"),
    71: ("none", "positive review"),
    72: ("difficulty_combat", "complaint about gunplay"),
    73: ("servers", "complaint about communication errors not yet fixed"),
    74: ("none", "positive review"),
    75: ("performance", "game can be laggy"),
    76: ("worse_than_predecessor;price_value;servers;multiplayer;bugs", "no community, no server browser, not worth price, each game iteration worse, bugs, dead multiplayer and community"),
    77: ("story;difficulty_combat", "poor character dialogue, brainless combat"),
    78: ("controls_ui;boring_grind;story", "poor story, boring gameplay, UI UX needs work"),
    79: ("none", "positive review anticipating full release"),
    80: ("none", "vague complaint"),
    81: ("bugs;lost_saves", "known bug wiped undid progress"),
    82: ("politics", "racist content in game"),
    83: ("wont_launch;controls_ui;boring_grind;graphics_audio;difficulty_combat", "difficult progression, grindy progression, graphics lacking optimization, some buttons stop working, difficult bosses"),
    84: ("none", "positive review"),
    85: ("dlc;updates_support", "updates took good things out, content behind DLCs"),
    86: ("difficulty_combat;boring_grind", "combat not engaging, busy work boring"),
    87: ("wont_launch;controls_ui", "compatibility issues with steam deck, UI may be too small"),
    88: ("boring_grind;accounts_drm", "slow paced, long duration events, scam"),
    89: ("none", "positive review"),
    90: ("updates_support;controls_ui;performance;crash", "worse with every update, UI is bad, poor performance downloading maps, crashes often"),
    91: ("story", "dialogue which feels AI generated"),
    92: ("performance", "not optimized well"),
    93: ("none", "postive review"),
    94: ("boring_grind;monetization", "boring after reaching max level, lots of content requires paying"),
    95: ("boring_grind", "progression feels a bit slow"), 
    96: ("price_value", "refunded"),
    97: ("wont_launch;bugs", "some bugs, not great controller support"),
    98: ("boring_grind;price_value", "not worth price, boring gameplay"),
    99: ("none", "positive review"),
    100: ("controls_ui;wont_launch", "no support for controller, no UI navigation"),
    101: ("controls_ui;tutorial", "not colorblind friendly (quality of life feature), gameplay difficult to understand"),
    102: ("bugs;story;worse_than_predecessor", "broken, poor dialogue, complaint about remaster, not original"),
    103: ("monetization;controls_ui;price_value", "lots of content needs to be bought, annoying UI, not worth price"),
    104: ("boring_grind", "grind to get good gear, behind after missing first day"),
    105: ("performance;updates_support", "game not optimized, development decided to scrap certain key feature"),
    106: ("graphics_audio;controls_ui", "poor graphics, camera"),
    107: ("none", "general complaint"),
    108: ("controls_ui;graphics_audio;worse_than_predecessor;performance;price_value", "worse than predecessor, game runs poorly, poor audio, poor UI, don't waste money"),
    109: ("difficulty_combat;price_value", "difficulty of enemies, not worth the price"),
    110: ("updates_support", "decision to ban mechanics and cosmetics"),
    111: ("boring_grind;controls_ui;monetization", "busywork, menu bloat, microsactions"),
    112: ("boring_grind;story;difficulty_combat", "boring game, flat story, combat wears out"),
    113: ("wont_launch", "game doesn't work"),
    114: ("multiplayer;servers;boring_grind;price_value", "poor matchmaking, server disconnects are an issue, game lacking progression, not worth price"),
    115: ("difficulty_combat", "game is poor for certain game role"),
    116: ("crash;controls_ui;price_value;bugs;multiplayer;graphics_audio", "game crashes, poor menu, coop feature seems forced, bugs, not worth price"),
    117: ("controls_ui;boring_grind", "horrible UI, complaint about game content/mechanics"),
    118: ("bugs;worse_than_predecessor", "bugs, not the best of the series"),
    119: ("monetization", "pay to win"),
    120: ("story", "annoying companions"),
    121: ("none", "positive review"),
    122: ("worse_than_predecessor;difficulty_combat;price_value", "certain mechanics worse than predecessor, poor combat, worth buying when cheap"),
    123: ("bugs", "some bugs"),
    124: ("boring_grind", "grind for resources"),
    125: ("none", "general negative"),
    126: ("story;boring_grind", "could have deeper dive into world, interactions, and gameplay"),
    127: ("price_value;boring_grind", "bored, refunded"),
    128: ("difficulty_combat;cheaters_toxicity", "gives unfair buffs to certain side, no management of toxicity, cheaters"),
    129: ("bugs;graphics_audio", "some bugs, bosses/enemies don't look great"),
    130: ("none", "general complaint"),
    131: ("none", "neither good or bad review"),
    132: ("none", "positive review"),
    133: ("boring_grind;cheaters_toxicity", "grindy game, no management of cheaters or toxicity"),
    134: ("controls_ui;boring_grind;price_value;story;difficulty_combat", "worth a try if on sale, story isn't perfect, poor audio, combat can be tedious, boring, difficult platforming"),
    135: ("lost_saves", "no option for saving or quiting"),
    136: ("none", "postive review"),
    137: ("servers;graphics_audio", "difficulty joining while online, forced aa blurry game?"),
    138: ("tutorial;bugs", "feels punished for not following tutorial exactly, bugs within tutorial that bricks the game"),    
    139: ("boring_grind", "grind long hours to unlock market"),
    140: ("lost_saves", "game resets every new in game day"),
    141: ("boring_grind", "only complaint is about the game being short"),
    142: ("boring_grind", "gets boring, game content doesn't really try anything new, boseses aren't unique"),
    143: ("price_value", "complaint about paying"),
    144: ("not_english", "review in different language"),
    145: ("difficulty_combat", "always out of ammo"),
    146: ("lost_saves;crash", "game crashed, resulting in a lost save"),
    147: ("difficulty_combat", "difficulty makes game frustrating"),
    148: ("difficulty_combat;worse_than_predecessor", "key combat mechanic ruined from last game"),
    149: ("boring_grind;performance", "game lags despite its size, bit boring"),
    150: ("tutorial;performance", "hard to learn everything, framerate struggles"),
    151: ("none", "positive review"),
    152: ("none", "neither positive or negative"),
    153: ("difficulty_combat;price_value;updates_support", "mechanic made game difficult and frustrating, wants money back, developer blocked tooling in game"),
    154: ("bugs;performance", "some bugs, small performance issues"),
    155: ("boring_grind", "game content could be more complex, simple moves"),
    156: ("none", "positive review"),
    157: ("none", "positive review"),
    158: ("story;performance;updates_support;boring_grind;difficulty_combat;worse_than_predecessor", "meh story, bad performance, worse after every update, poor combat, worse than previous games"),
    159: ("worse_than_predecessor;story;difficulty_combat;bugs;boring_grind;price_value", "worse than previous games, poor story, glitches, not worth value, boring"),
}

In [59]:
done = sorted(test_answers)
bad = {i: c for i in done for c in test_answers[i][0].split(";") if c not in CODES}
assert not bad, f"unknown codes: {bad}"
lab = pd.read_csv(TEST_LABEL, encoding="utf-8-sig", dtype={"topics": str, "note": str})
lab.loc[done, "topics"] = [test_answers[i][0] for i in done]
lab.loc[done, "note"] = [test_answers[i][1] for i in done]
lab.to_csv(TEST_LABEL, index=False, encoding="utf-8-sig")
print(f"{lab.topics.notna().sum()} of {len(lab)} test reviews labeled")

160 of 160 test reviews labeled
